# Download dos Áudios e Transcrições

Importação de bibliotecas

In [25]:
import pandas as pd
import warnings
import requests
import os
import subprocess
import whisper
from tqdm import tqdm
import shutil
warnings.filterwarnings('ignore')

Df com informações dos vídeos

In [26]:
df = pd.read_csv('../data/amostra_comparacao_transcricao.csv')
df = df.rename(columns={'videoId': 'video_id'})

In [27]:
df.columns

Index(['video_id', 'title', 'transcription'], dtype='object')

In [28]:
df['video_id'].nunique()

19

# Filtragem dos vídeos disponíveis

O primeiro passo que precisamos fazer é filtrar os vídeos que estão no nosso dataset para sabermos quais ainda estão disponíveis no Youtube.

In [29]:
def check_video_availability(video_id):
    url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
    response = requests.get(url)
    
    return response.status_code == 200

In [30]:
def save_progress(valid_videos, invalid_videos):
    """Salva os vídeos válidos e inválidos em arquivos CSV separados."""
    valid_videos.to_csv("../data/filtrated_data_log/valid_videos.csv", index=False)
    invalid_videos.to_csv("../data/filtrated_data_log/invalid_videos.csv", index=False)

In [31]:
def load_progress():
    """Carrega o progresso de vídeos válidos e inválidos (se houver)."""
    try:
        valid_videos = pd.read_csv("../../filtrated_data_log/valid_videos.csv")
    except FileNotFoundError:
        valid_videos = pd.DataFrame(columns=["video_id"])
    
    try:
        invalid_videos = pd.read_csv("../../filtrated_data_log/invalid_videos.csv")
    except FileNotFoundError:
        invalid_videos = pd.DataFrame(columns=["video_id"])
    
    return valid_videos, invalid_videos

In [32]:
valid_videos, invalid_videos = load_progress()

In [33]:
for video_id in tqdm(df['video_id'].drop_duplicates()):
    if video_id not in valid_videos["video_id"].values and video_id not in invalid_videos["video_id"].values:
        status = check_video_availability(video_id)
        
        if status is not None:
            new_row = pd.DataFrame({"video_id": [video_id]})
            if status:
                valid_videos = pd.concat([valid_videos, new_row], ignore_index=True)
                #print(f"O vídeo {video_id} está disponível.")
            else:
                invalid_videos = pd.concat([invalid_videos, new_row], ignore_index=True)
                #print(f"O vídeo {video_id} foi removido ou está indisponível.")
        
        save_progress(valid_videos, invalid_videos)

100%|██████████| 19/19 [00:06<00:00,  2.72it/s]


# Filtrar todos os dados dos vídeos disponíveis

O objetivo desse bloco é verificar as informações dos vídeos que estão disponíveis, para isso, eu deixei o dataset com os dados dos vídeos, como título, descrição e etc. Com isso, conseguimos fazer análises sobre a quantidade de vídeos favoráveis/contrários entre outras.

In [34]:
# df_dados = pd.read_csv('../raw_data/videos_info_clean.csv')
df_filtro = pd.read_csv('../data/filtrated_data_log/valid_videos.csv')

In [35]:
df['video_id'] = df['video_id'].astype(str)
df_filtro['video_id'] = df_filtro['video_id'].astype(str)

In [36]:
dados_filtrados = df[df['video_id'].isin(df_filtro['video_id'])]

In [37]:
dados_filtrados.to_csv('../data/filtrated_data_log/videos_filtrados.csv', index=False)
dados_filtrados.head()

,video_id,title,transcription
0,w22lrhGXy_I,FELCA O MELHOR XERIFE DO MM2! #roblox,"Eu sou polícia, mano. Não vou ser polícia e vo..."
1,wpRkmL6Hbbo,Dep Federal Jadyel Alencar estará no Fantástic...,Estaremos hoje no Fantástico para falar do pro...
2,ZzrG8BgHoZ0,As primeiras impressões sobre o ECA Digital | ...,E esse vai ser um dos assuntos da nossa coluna...
3,glr13drvkhE,"Além do tempo, o conteúdo também importa!",Deixa eu te mostrar o que seu filho pode estar...
4,pmmwmIJZ5_k,BASEUS WM01 É BOM? O MELHOR CUSTO-BENEFÍCIO EM...,O design dos fones de ouvido permite obter um ...


# Fazendo o download dos vídeos

In [38]:
# Configurações
audio_folder = "../audios"
log_file = "../download_transcription_progress/download_progress.log"
csv_file = "../data/filtrated_data_log/valid_videos.csv"

In [39]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [40]:
df = pd.read_csv(csv_file)
video_ids = df["video_id"].astype(str).tolist()

In [41]:
completed_videos = load_progress()

In [42]:
for video_id in video_ids:
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")

    # Se já foi baixado, pula
    if video_id in completed_videos or os.path.exists(audio_path):
        print(f"Áudio já baixado: {audio_path}")
        continue

    url = f"https://www.youtube.com/watch?v={video_id}"
    print(f"Baixando áudio de {url}...")

    try:
        result = subprocess.run(
            ["yt-dlp", "-f", "bestaudio", "--extract-audio", "--audio-format", "mp3",
             "-o", f"{audio_folder}/%(id)s.%(ext)s", url],
            check=True, text=True, capture_output=True
        )
        print(f"Download concluído: {audio_path}")
        save_progress(video_id)

    except subprocess.CalledProcessError as e:
        print(f"Erro ao baixar {video_id}: {e.stderr}")

print("Processo de download concluído!")

Áudio já baixado: ../audios/w22lrhGXy_I.mp3
Baixando áudio de https://www.youtube.com/watch?v=wpRkmL6Hbbo...
Erro ao baixar wpRkmL6Hbbo: WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one
ERROR: [youtube] wpRkmL6Hbbo: This video is not available

Áudio já baixado: ../audios/ZzrG8BgHoZ0.mp3
Baixando áudio de https://www.youtube.com/watch?v=glr13drvkhE...
Erro ao baixar glr13drvkhE: WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp

# Transcrever os Áudios

In [43]:
audio_folder = "../audios/"
transcription_folder = "../transcriptions"
log_file = "../download_transcription_progress/progresso_transcricao.log"

In [44]:
df = pd.read_csv("../data/amostra_comparacao_transcricao.csv")
df = df.rename(columns={'videoId': 'video_id'})
video_ids = df["video_id"].drop_duplicates().tolist()

In [45]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [46]:
completed_videos = load_progress()

In [47]:
modelo = whisper.load_model("small").to("cuda")

In [48]:
for video_id in df['video_id'].drop_duplicates():
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")
    output_text = os.path.join(transcription_folder, f"{video_id}.txt")
    zip_audio_base = os.path.join(audio_folder, video_id)  # Sem extensão

    # Se já foi transcrito, pula
    if video_id in completed_videos or os.path.exists(output_text):
        print(f"Transcrição já feita para {video_id}")
        continue

    if not os.path.exists(audio_path):
        print(f"Áudio não encontrado para {video_id}, pulando...")
        continue

    print(f"Transcrevendo {audio_path}...")

    try:
        result = modelo.transcribe(audio_path, language="portuguese", fp16=False)

        with open(output_text, "w", encoding="utf-8") as f:
            f.write(result["text"])

        save_progress(video_id)
        print(f"Transcrição salva: {output_text}")

        # Compacta apenas o arquivo MP3
        shutil.make_archive(zip_audio_base, 'zip', audio_folder, f"{video_id}.mp3")
        print(f"Áudio compactado: {zip_audio_base}.zip")
        
        os.remove(audio_path)
        print(f"Áudio excluído: {audio_path}")

    except Exception as e:
        print(f"Erro ao transcrever {video_id}: {str(e)}")

print("Processo de transcrição concluído!")

Transcrevendo ../audios/w22lrhGXy_I.mp3...
Transcrição salva: ../transcriptions/w22lrhGXy_I.txt
Áudio compactado: ../audios/w22lrhGXy_I.zip
Áudio excluído: ../audios/w22lrhGXy_I.mp3
Áudio não encontrado para wpRkmL6Hbbo, pulando...
Transcrição já feita para ZzrG8BgHoZ0
Áudio não encontrado para glr13drvkhE, pulando...
Transcrevendo ../audios/pmmwmIJZ5_k.mp3...
Transcrição salva: ../transcriptions/pmmwmIJZ5_k.txt
Áudio compactado: ../audios/pmmwmIJZ5_k.zip
Áudio excluído: ../audios/pmmwmIJZ5_k.mp3
Transcrevendo ../audios/diiNz7u3QEs.mp3...
Transcrição salva: ../transcriptions/diiNz7u3QEs.txt
Áudio compactado: ../audios/diiNz7u3QEs.zip
Áudio excluído: ../audios/diiNz7u3QEs.mp3
Transcrevendo ../audios/0kiWV3FJik0.mp3...
Transcrição salva: ../transcriptions/0kiWV3FJik0.txt
Áudio compactado: ../audios/0kiWV3FJik0.zip
Áudio excluído: ../audios/0kiWV3FJik0.mp3
Transcrevendo ../audios/-7n-QWop-4M.mp3...
Transcrição salva: ../transcriptions/-7n-QWop-4M.txt
Áudio compactado: ../audios/-7n-QWop-4